In [13]:
import warnings
warnings.filterwarnings('ignore')

import argparse
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, accuracy_score, precision_score, recall_score, f1_score

import optuna
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


In [16]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('submission_example.csv')

# 2. 전처리 함수
# pdays, 로그 변환, 파생 변수
# month 문자열 -> 숫자 매핑
month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
for df in [train, test]:
    # pdays
    df['pdays'] = df['pdays'].replace(-1, 999)
    # 로그 변환
    df['balance_log'] = np.log1p(df['balance'])
    df['duration_log'] = np.log1p(df['duration'])
    # 캠페인당 duration
    df['duration_per_campaign'] = df['duration'] / (df['campaign'] + 1)
    # month 숫자 컬럼 추가
    df['month_num'] = df['month'].str.lower().map(month_map)
    # day_of_year
    df['day_of_year'] = df['day'] + (df['month_num'] - 1) * 30
    # 성공 여부 파생
    df['poutcome_success'] = (df['poutcome'] == 'success').astype(int)

In [17]:
# 3. 범주형 인코딩
cols = ['job','marital','contact','poutcome','education','default','housing','loan','month']
for col in cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

In [18]:
# 4. 데이터 분할
X = train.drop(['ID','label'], axis=1)
y = train['label']
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


In [19]:
# 5. 파이프라인
# duration만 PowerTransformer, 나머지 그대로
numeric_cols = [c for c in X.columns if c != 'duration']
preprocessor = ColumnTransformer([
    ('pt_dur', PowerTransformer(), ['duration']),
    ('pass', 'passthrough', numeric_cols)
])

X_train_p = preprocessor.fit_transform(X_train)
X_val_p   = preprocessor.transform(X_val)

# 컬럼명 가져오기
feature_names = preprocessor.get_feature_names_out()


In [20]:

# 6. 특성 선택 (SequentialFS)
from sklearn.feature_selection import SequentialFeatureSelector
base_clf = LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1)
sfs = SequentialFeatureSelector(base_clf, n_features_to_select=10, direction='forward', scoring='f1', cv=5, n_jobs=-1)
sfs.fit(X_train_p, y_train)
sel_feats = feature_names[sfs.get_support()]
X_train_s = pd.DataFrame(X_train_p, columns=feature_names)[sel_feats]
X_val_s   = pd.DataFrame(X_val_p, columns=feature_names)[sel_feats]

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted w

In [21]:
# 7. Optuna로 LGBM 튜닝

def opt_lgb(trial):
    params = {
        'class_weight':'balanced',
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3,1.0),
        'n_estimators': trial.suggest_int('n_estimators',100,1000),
        'max_depth': trial.suggest_int('max_depth',3,12),
        'subsample': trial.suggest_uniform('subsample',0.5,1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree',0.5,1.0),
        'min_child_samples': trial.suggest_int('min_child_samples',20,200)
    }
    clf = LGBMClassifier(**params, random_state=42, verbose=-1)
    return cross_val_score(clf, X_train_s, y_train, scoring='f1', cv=5, n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(opt_lgb, n_trials=30)
best_params = study.best_params
print('LGBM Best Params:', best_params)


[I 2025-05-30 05:08:24,377] A new study created in memory with name: no-name-4d43109f-52f2-4500-8832-396bbe69ef64
[I 2025-05-30 05:08:28,248] Trial 0 finished with value: 0.6020954959066216 and parameters: {'learning_rate': 0.06512694331292747, 'n_estimators': 999, 'max_depth': 3, 'subsample': 0.8682695541178378, 'colsample_bytree': 0.890288473452336, 'min_child_samples': 84}. Best is trial 0 with value: 0.6020954959066216.
[I 2025-05-30 05:08:30,852] Trial 1 finished with value: 0.5946819597358459 and parameters: {'learning_rate': 0.049703535745408284, 'n_estimators': 408, 'max_depth': 4, 'subsample': 0.5012752435846946, 'colsample_bytree': 0.8617076887995382, 'min_child_samples': 139}. Best is trial 0 with value: 0.6020954959066216.
[I 2025-05-30 05:08:39,262] Trial 2 finished with value: 0.6011674129269174 and parameters: {'learning_rate': 0.02347943385191336, 'n_estimators': 437, 'max_depth': 9, 'subsample': 0.5947410283317385, 'colsample_bytree': 0.604265666853882, 'min_child_samp

LGBM Best Params: {'learning_rate': 0.11800254358033795, 'n_estimators': 282, 'max_depth': 7, 'subsample': 0.673129707501364, 'colsample_bytree': 0.7930816621363459, 'min_child_samples': 109}


In [24]:
import lightgbm as lgbm
# 8. 앙상블 모델 정의
rf  = RandomForestClassifier(n_estimators=200, max_depth=7, class_weight='balanced', random_state=42)
cb  = CatBoostClassifier(verbose=0, auto_class_weights='Balanced', random_state=42)
stk = StackingClassifier(estimators=[('lgbm',lgbm),('rf',rf),('cb',cb)], final_estimator=LogisticRegression(class_weight='balanced',max_iter=1000,random_state=42), cv=5, n_jobs=-1, passthrough=True)

# 앙상블 학습
stk.fit(X_train_s, y_train)


TypeError: Cannot clone object. You should provide an instance of scikit-learn estimator instead of a class.

In [ ]:
# 9. 임계값 탐색 + 평가
for name, model in [('LGBM', lgbm), ('Ensemble', stk)]:
    prob = model.predict_proba(X_val_s)[:,1]
    precision, recall, thresh = precision_recall_curve(y_val, prob)
    f1s = 2*precision*recall/(precision+recall+1e-9)
    opt_t = thresh[f1s.argmax()]
    y_pred = (prob>opt_t).astype(int)
    print(f"{name} F1:", f1_score(y_val, y_pred).round(4), "thr=", round(opt_t,3))

In [ ]:
# 10. 테스트 예측 및 제출
X_test_p = preprocessor.transform(test.drop('ID',axis=1))
X_test_s = pd.DataFrame(X_test_p, columns=feature_names)[sel_feats]
sub_pred = (stk.predict_proba(X_test_s)[:,1] > opt_t).astype(int)
submission['label'] = sub_pred
submission.to_csv('LAST(0900)_submission.csv', index=False)